# 03 - uncertainty hard-case mining과 Overall 검산

**학습 목표**: 여러 stochastic OCR 결과의 pairwise consistency로 hard sample을 고르고, 논문의 OmniDocBench Overall 공식을 검산합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리 `itertools`만 사용합니다.

In [ ]:
from itertools import combinations
# combinations를 사용하면 T개 추론의 중복 없는 모든 pair를 정확히 한 번씩 평가합니다.

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[-1] + 1, prev[j] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def similarity(a, b):
    return 1.0 - edit_distance(a, b) / max(1, len(a), len(b))

def consistency(outputs):
    scores = [similarity(a, b) for a, b in combinations(outputs, 2)]
    return sum(scores) / len(scores)

samples = {
    'easy': ['TOTAL 120', 'TOTAL 120', 'TOTAL 120'],
    'hard': ['TOTAL 120', 'T0TAL 120', 'TOTAL 12O'],
}
tau = 0.95
for name, outputs in samples.items():
    c = consistency(outputs)
    weight = 1 + 2.0 * (1 - c)  # beta=2인 toy weighting
    print(name, 'C(x)=', round(c, 3), 'hard=', c < tau, 'weight=', round(weight, 3))

def overall(text_edit, formula_cdm, table_teds):
    return ((1 - text_edit) * 100 + formula_cdm + table_teds) / 3

score = overall(0.028, 91.92, 91.00)
print('MinerU-Diffusion w/ GT Layout Overall:', round(score, 2))
assert round(score, 2) == 93.37
assert consistency(samples['hard']) < consistency(samples['easy'])


실제 논문은 layout에 PageIoU, formula에 CDM, table에 TEDS처럼 task별 `S`를 사용합니다. 문자열 similarity 하나로 모든 구조를 평가한 이 toy는 curriculum 선택 원리만 설명합니다.